# Tutorial 3: Model Diagnosis

After training a deepSCENIC model, the critical question is: **did the model learn biologically meaningful regulatory patterns?**

This tutorial covers how to extract and visualize the model's learned representations to assess whether it has captured cell-type-specific transcription factor (TF) activity. We'll look at several embedding spaces and discuss what good (and bad) results look like.

## What We're Looking For

A well-trained deepSCENIC model should learn that different cell types have distinct TF activity profiles. The clearest signal of this is whether cells cluster by type in the model's latent embedding spaces. If cells of the same type cluster together, the model has learned to distinguish them based on their regulatory programs.

## Setup

First, let's import the required packages. We'll use scanpy for UMAP visualization since deepSCENIC stores embeddings in the standard `mdata.obsm` format that scanpy expects.

In [1]:
import torch
import scanpy as sc

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

deepSCENIC version: 0.1.0
Using device: cuda


## Load Model and Data

We load the preprocessed MuData from {doc}`01_data_preparation` and the trained model from {doc}`02_training`. Make sure your MuData includes cell annotations (like cell type or cell state) that you can use to color the UMAP plots.

In [2]:
import os

data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "Data directory not found"

# Load preprocessed MuData
mdata = ds.read(data_dir + "preprocessed_data.h5mu")

# Load trained model
model = ds.tl.load_model(data_dir + "weights/converted_model.pt", device=device)

# Verify cell annotations are available (if you have them)
print(f"\nCell annotations: {list(mdata.obs.columns)}")
if "cell_state" in mdata.obs:
    print(f"Cell states: {mdata.obs['cell_state'].value_counts().to_dict()}")

deepscenic.io - INFO - Loaded ../../../../data/deepSCENIC/MM_lines/preprocessed_data.h5mu: 936 cells, 12156 genes, 281820 regions
deepscenic.tl - INFO - Loaded model from ../../../../data/deepSCENIC/MM_lines/weights/converted_model.pt: 11717 genes, 1386 TFs, 273545 regions



Cell annotations: ['cell_line', 'cell_state', 'split']
Cell states: {'INT': 358, 'MEL': 318, 'MES': 260}


```{tip}
If you encounter GPU memory errors during inference, reduce the `batch_size` parameter in `ds.tl.to_latent()`. The default (256) works for most cases, but very large datasets may require smaller batches.
```

## Extract Model Embeddings

The deepSCENIC model produces several intermediate representations as it transforms TF expression into gene expression predictions. Each of these embeddings captures a different aspect of gene regulation:

| Embedding | Shape | Description |
|-----------|-------|-------------|
| **z_tf** | (n_cells, n_tfs) | Latent TF activity after the encoder. This is the model's learned representation of each TF's regulatory state in each cell. |
| **enh_act** | (n_cells, n_regions) | Enhancer/region activity, computed as z_tf @ E1. Represents how active each regulatory region is based on TF binding predictions. |
| **z_rna** | (n_cells, n_genes) | Gene regulatory signal, computed as enh_act @ E2. The predicted regulatory input to each gene before the decoder transforms it to expression. |
| **rna_rec** | (n_cells, n_genes) | Reconstructed gene expression from the decoder. |
| **atac_rec** | (n_cells, n_regions) | Reconstructed chromatin accessibility from the decoder. |

The {func}`~deepscenic.tl.to_latent` function extracts all of these in one forward pass and stores them in `mdata.obsm` with the prefix `X_deepscenic_`. This format is compatible with scanpy's visualization functions.

In [ ]:
# Extract all embeddings in one pass and store in mdata.obsm
# Use batch_size to avoid GPU memory issues
ds.tl.to_latent(model, mdata, batch_size=64)

# Embeddings are now stored in mdata.obsm with X_deepscenic_ prefix
print("Extracted embeddings (stored in mdata.obsm):")
for key in mdata.obsm.keys():
    if key.startswith("X_deepscenic_"):
        print(f"  {key}: {mdata.obsm[key].shape}")

## Visualizing the TF Activity Space

The most important diagnostic is visualizing the **TF activity space (z_tf)**. This embedding captures the model's learned representation of transcription factor regulatory states.

### Why z_tf is the Key Diagnostic

The z_tf embedding is the best indicator of whether your model learned meaningful biology because:

1. **It's the core learned representation**: The encoder transforms raw TF expression into this latent space. If the model learned well, z_tf should capture regulatory states rather than just expression levels.

2. **It's interpretable**: Each dimension corresponds to a specific TF, so you can directly examine which TFs drive cell-type differences.

3. **It's the right dimensionality**: With ~1,000-2,000 TFs, z_tf is low-dimensional enough to visualize directly with UMAP without requiring PCA first.

### What to Expect

For a well-trained model on the MM_lines dataset, you should see:
- **MEL (melanocytic)** and **MES (mesenchymal)** cells forming distinct clusters
- **INT (intermediate)** cells positioned between them, reflecting their transitional state
- Clear separation by cell state, not just by cell line or batch

## Other Embedding Spaces (Optional)

While z_tf is the primary diagnostic, you may also want to examine the other embeddings:

### Gene Regulatory Signal (z_rna)

The z_rna embedding represents the predicted regulatory input to each gene. It's computed by multiplying enhancer activity by the E2 (region→gene) matrix. Visualizing z_rna can help verify that the regulatory signal is consistent with your cell type expectations.

```python
# z_rna is high-dimensional (~12k genes), so we need PCA first
import anndata as ad

adata_rna = ad.AnnData(mdata.obsm["X_deepscenic_z_rna"])
adata_rna.obs = mdata.obs.copy()
sc.pp.pca(adata_rna, n_comps=50)
sc.pp.neighbors(adata_rna)
sc.tl.umap(adata_rna)
sc.pl.umap(adata_rna, color="cell_state", title="Gene Regulatory Signal (z_rna)")
```

### Enhancer Activity (enh_act)

The enh_act embedding is typically very high-dimensional (~280k regions) and can be noisy. We generally recommend focusing on z_tf and z_rna for model diagnosis, and using the cell-type-specific enhancer analysis in {doc}`04_grn_analysis` for detailed enhancer interpretation.

```{note}
The reconstruction embeddings (rna_rec, atac_rec) are less useful for diagnosis since they're just the decoded outputs. The latent spaces (z_tf, z_rna) better reflect what the model has learned.
```

In [ ]:
# Compute UMAP on the TF activity embedding
# Since z_tf is already low-dimensional (~1400 TFs), we can use it directly
sc.pp.neighbors(mdata, use_rep="X_deepscenic_z_tf")
sc.tl.umap(mdata)

# Visualize colored by cell state
sc.pl.umap(mdata, color="cell_state", title="TF Activity Space (z_tf)")

### Comparing to Raw TF Expression

A useful sanity check is comparing the learned z_tf representation to a UMAP of the raw TF expression. If the model learned meaningful regulatory patterns, the z_tf UMAP should show cleaner separation than raw expression (since it captures regulatory state, not just expression levels).

Let's visualize both side-by-side:

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt

# Extract raw TF expression for comparison
tf_mask = mdata['rna'].var['is_tf'].values
tf_expr = mdata['rna'].X[:, tf_mask]
if hasattr(tf_expr, 'toarray'):
    tf_expr = tf_expr.toarray()

# Create AnnData for raw TF expression and compute UMAP
adata_tf_raw = ad.AnnData(tf_expr)
adata_tf_raw.obs = mdata.obs.copy()
sc.pp.neighbors(adata_tf_raw)
sc.tl.umap(adata_tf_raw)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw TF expression UMAP
sc.pl.umap(adata_tf_raw, color="cell_state", ax=axes[0], show=False, title="Raw TF Expression")

# Learned z_tf UMAP (already computed on mdata)
sc.pl.umap(mdata, color="cell_state", ax=axes[1], show=False, title="Learned TF Activity (z_tf)")

plt.tight_layout()
plt.show()

Both plots should show cell-type structure, but the learned z_tf representation often shows:
- **Tighter clusters**: The encoder learns to filter noise and extract the regulatory signal
- **Better separation**: The VAE regularization encourages a well-structured latent space
- **More meaningful intermediate states**: Transition cells may be positioned more biologically appropriately

If the two plots look nearly identical, the model may not be learning much beyond the raw expression. If z_tf looks worse than raw expression, something may have gone wrong during training.

## Interpreting Your UMAP Results

### Good Signs ✓

A well-trained model will show:

- **Distinct clusters matching cell type annotations**: Cells of the same type should group together
- **Biologically meaningful structure**: For example, intermediate states should appear between the extremes (like INT between MEL and MES)
- **Consistent patterns across z_tf and z_rna**: Both spaces should show similar cell-type organization
- **No obvious batch effects**: Clusters should be driven by biology, not by sample or batch

### Warning Signs ⚠

If you see these patterns, something may have gone wrong:

| Pattern | Possible Cause | Solution |
|---------|----------------|----------|
| **Random mixing of cell types** | Model didn't learn meaningful TF patterns | More training epochs, check TF marking |
| **Clustering by batch/sample instead of cell type** | Batch effects dominating | Consider batch correction, check data quality |
| **Single blob with no structure** | Training collapsed or too few epochs | Increase epochs, check learning rate |
| **Very noisy, fragmented clusters** | Overfitting or data quality issues | Reduce model complexity, check preprocessing |

### Troubleshooting Tips

If your results look suboptimal:

1. **Check your TF list**: Are the right genes marked as TFs? Run `mdata['rna'].var['is_tf'].sum()` to verify.
2. **Try more training**: Early stopping may have triggered too soon.
3. **Examine the training curves**: Use `ds.pl.loss_curves(model)` if available (see next section).
4. **Verify data preprocessing**: Ensure RNA is log-normalized and ATAC is imputed.

## Training Curves (For Newly Trained Models)

If you trained the model yourself using `ds.tl.train()` (rather than loading a converted legacy model), you can examine the training history to assess convergence.

```python
# Plot training curves (only available for models trained with ds.tl.train())
ds.pl.loss_curves(model)
```

### What to Look For

- **Decreasing loss**: Both train and test losses should decrease over epochs
- **No large gap between train/test**: A large gap suggests overfitting
- **Stable convergence**: Loss should stabilize, not oscillate wildly
- **All loss components behaving**: Check RNA loss, ATAC loss, KL divergence, and sparsity terms

```{note}
Legacy models loaded with `ds.tl.load_legacy_model()` don't have training history, so this function won't work for them.
```

## TF Activity Profiles by Cell Type

Once you've confirmed good clustering in the z_tf space, you can dive deeper into which TFs drive the cell-type differences. A clustered heatmap of TF activity scores reveals the regulatory programs that distinguish each cell type.

### Computing TF Activity Scores

The TF activity score for each cell type is computed by:
1. Identifying active enhancers per cell type (using Otsu thresholding on enhancer activity)
2. Weighting TF activity by their binding to these active enhancers (via the E1 matrix)

This gives a biologically meaningful measure of how important each TF is for each cell type's regulatory program.

In [ ]:
# Compute TF activity scores for visualization
enh_activity = ds.tl.compute_celltype_enhancer_activity(model, mdata, "cell_state")
active_enh = ds.tl.identify_active_enhancers(enh_activity)
tf_scores = ds.tl.compute_tf_activity_scores(model, mdata, "cell_state", active_enh)

# Show top 50 most variable TFs across cell states
ds.pl.tf_activity_clustermap(tf_scores, xlabel="cell_state", top_n_tfs=50)

In the heatmap above, each row is a TF and each column is a cell type. TFs are clustered by their activity patterns, so TFs with similar cell-type-specificity appear together. Look for:

- **Cell-type-specific TFs**: TFs that are highly active in one cell type but not others
- **Shared TFs**: TFs active across multiple cell types (these may be housekeeping regulators)
- **Known biology**: For melanoma, you might expect MITF-family TFs to be active in MEL cells, and AP-1/inflammatory TFs in MES cells

This analysis is explored in more detail in {doc}`04_grn_analysis`.

## Summary

In this tutorial, we've covered:

1. **Extracting embeddings** from the trained model using `ds.tl.to_latent()`, which stores results in `mdata.obsm`
2. **Visualizing the TF activity space (z_tf)** as the primary model diagnostic
3. **Comparing to raw expression** to verify the model learned beyond just expression patterns
4. **Interpreting UMAP results** to assess whether the model learned meaningful biology
5. **Examining TF activity profiles** to identify cell-type-specific regulatory programs

## Next Steps

Now that you've validated your model:

- **{doc}`04_grn_analysis`**: Extract and analyze the learned gene regulatory network, including TF→region→gene cascades
- **{doc}`05_perturbation_analysis`**: Simulate in-silico TF knockouts and overexpression to predict downstream effects